# 🧠 Studi Kasus Klasifikasi — Breast Cancer Wisconsin
## Menggunakan Artificial Neural Network (ANN) dengan TensorFlow/Keras

---

### 📌 Deskripsi Proyek
Pada studi kasus ini, kita akan melakukan **klasifikasi biner** untuk mendeteksi apakah tumor bersifat **Malignant (ganas)** atau **Benign (jinak)** menggunakan dataset publik **Breast Cancer Wisconsin** dari `sklearn.datasets`.

### 🎯 Tujuan
- Membangun model ANN (Artificial Neural Network) untuk klasifikasi
- Melakukan prediksi terhadap data uji
- Menganalisis performa model

### 📦 Dataset
- **Sumber**: sklearn.datasets (UCI Machine Learning Repository)
- **Jumlah Sampel**: 569
- **Jumlah Fitur**: 30
- **Kelas**: 2 (Malignant = 0, Benign = 1)


## 1. 📥 Import Library

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, roc_auc_score, roc_curve)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

import warnings
warnings.filterwarnings('ignore')

print('✅ TensorFlow version:', tf.__version__)
print('✅ Semua library berhasil diimport!')

## 2. 📊 Load dan Eksplorasi Dataset

In [ ]:
# Load dataset
data = load_breast_cancer()

# Buat DataFrame
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target
df['diagnosis'] = df['target'].map({0: 'Malignant', 1: 'Benign'})

print('📋 Informasi Dataset:')
print(f'   - Jumlah sampel  : {df.shape[0]}')
print(f'   - Jumlah fitur   : {df.shape[1] - 2}')
print(f'   - Kelas target   : {list(data.target_names)}')
print()
print('📈 Distribusi Kelas:')
print(df['diagnosis'].value_counts())
print()
df.head()

In [ ]:
# Cek missing values
print('🔍 Missing Values:')
print(df.isnull().sum().sum(), 'missing values ditemukan')
print()
print('📊 Statistik Deskriptif:')
df.describe().round(2)

In [ ]:
# Visualisasi distribusi kelas
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
counts = df['diagnosis'].value_counts()
axes[0].pie(counts, labels=counts.index, autopct='%1.1f%%',
            colors=['#FF6B6B', '#4ECDC4'], startangle=90,
            textprops={'fontsize': 13})
axes[0].set_title('Distribusi Kelas Target', fontsize=14, fontweight='bold')

# Bar chart
sns.countplot(data=df, x='diagnosis', palette=['#FF6B6B', '#4ECDC4'], ax=axes[1])
axes[1].set_title('Jumlah Sampel per Kelas', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Diagnosis')
axes[1].set_ylabel('Jumlah')
for p in axes[1].patches:
    axes[1].annotate(f'{int(p.get_height())}',
                     (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha='center', va='bottom', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Distribusi kelas: Dataset cukup seimbang')

In [ ]:
# Correlation heatmap (top 10 fitur)
top_features = df.corr()['target'].abs().sort_values(ascending=False)[1:11].index.tolist()

plt.figure(figsize=(12, 8))
corr_matrix = df[top_features + ['target']].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            mask=mask, linewidths=0.5, annot_kws={'size': 9})
plt.title('Correlation Heatmap — Top 10 Fitur vs Target', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. 🔧 Preprocessing Data

In [ ]:
# Pisahkan fitur dan label
X = data.data
y = data.target

# Split data: 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Normalisasi fitur dengan StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print('✅ Data berhasil diproses!')
print(f'   - Training set : {X_train_scaled.shape[0]} sampel')
print(f'   - Testing set  : {X_test_scaled.shape[0]} sampel')
print(f'   - Jumlah fitur : {X_train_scaled.shape[1]}')

## 4. 🧠 Membangun Model ANN (Artificial Neural Network)

In [ ]:
# Bangun arsitektur ANN
tf.random.set_seed(42)

model = keras.Sequential([
    # Input Layer
    layers.Input(shape=(30,)),

    # Hidden Layer 1
    layers.Dense(64, activation='relu', name='hidden_1'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),

    # Hidden Layer 2
    layers.Dense(32, activation='relu', name='hidden_2'),
    layers.BatchNormalization(),
    layers.Dropout(0.2),

    # Hidden Layer 3
    layers.Dense(16, activation='relu', name='hidden_3'),

    # Output Layer (sigmoid untuk klasifikasi biner)
    layers.Dense(1, activation='sigmoid', name='output')
], name='ANN_BreastCancer')

# Compile model
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

### Arsitektur ANN:
```
Input (30 fitur)
    ↓
Dense(64, ReLU) → BatchNorm → Dropout(0.3)
    ↓
Dense(32, ReLU) → BatchNorm → Dropout(0.2)
    ↓
Dense(16, ReLU)
    ↓
Dense(1, Sigmoid) → Output (0=Malignant, 1=Benign)
```

## 5. 🏋️ Training Model

In [ ]:
# Callbacks
early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=15, restore_best_weights=True
)
reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=7, min_lr=1e-6
)

# Training
history = model.fit(
    X_train_scaled, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

print(f'\n✅ Training selesai pada epoch: {len(history.history["loss"])}')

In [ ]:
# Visualisasi Training History
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(history.history['loss'], label='Train Loss', color='#FF6B6B', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Val Loss', color='#4ECDC4', linewidth=2)
axes[0].set_title('Training & Validation Loss', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Accuracy
axes[1].plot(history.history['accuracy'], label='Train Accuracy', color='#FF6B6B', linewidth=2)
axes[1].plot(history.history['val_accuracy'], label='Val Accuracy', color='#4ECDC4', linewidth=2)
axes[1].set_title('Training & Validation Accuracy', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('training_history.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. 🔮 Prediksi dan Evaluasi Model

In [ ]:
# Prediksi
y_pred_prob = model.predict(X_test_scaled, verbose=0).flatten()
y_pred      = (y_pred_prob >= 0.5).astype(int)

# Evaluasi
test_loss, test_acc = model.evaluate(X_test_scaled, y_test, verbose=0)
roc_auc = roc_auc_score(y_test, y_pred_prob)

print('=' * 50)
print('📊 HASIL EVALUASI MODEL ANN')
print('=' * 50)
print(f'✅ Test Accuracy : {test_acc:.4f} ({test_acc*100:.2f}%)')
print(f'✅ Test Loss     : {test_loss:.4f}')
print(f'✅ ROC-AUC Score : {roc_auc:.4f}')
print('=' * 50)
print()
print('📋 Classification Report:')
print(classification_report(y_test, y_pred, target_names=['Malignant', 'Benign']))

In [ ]:
# Visualisasi Confusion Matrix dan ROC Curve
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Malignant', 'Benign'],
            yticklabels=['Malignant', 'Benign'],
            linewidths=1, linecolor='white', annot_kws={'size': 16})
axes[0].set_title('Confusion Matrix', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Actual Label')
axes[0].set_xlabel('Predicted Label')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_pred_prob)
axes[1].plot(fpr, tpr, color='#FF6B6B', linewidth=2.5,
             label=f'AUC = {roc_auc:.4f}')
axes[1].plot([0, 1], [0, 1], 'k--', linewidth=1.5, label='Random Classifier')
axes[1].fill_between(fpr, tpr, alpha=0.1, color='#FF6B6B')
axes[1].set_title('ROC Curve', fontsize=14, fontweight='bold')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend(fontsize=12)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('evaluation_results.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. 🔍 Prediksi Data Baru (Contoh)

In [ ]:
# Prediksi beberapa sampel dari test set
print('🔮 Contoh Hasil Prediksi (10 sampel pertama dari test set):')
print('-' * 65)
print(f'{"No":>4} | {"Actual":>12} | {"Predicted":>12} | {"Confidence":>12} | {"Status"}')
print('-' * 65)

label_map = {0: 'Malignant', 1: 'Benign'}
for i in range(10):
    actual    = label_map[y_test[i]]
    predicted = label_map[y_pred[i]]
    prob      = y_pred_prob[i] if y_pred[i] == 1 else 1 - y_pred_prob[i]
    status    = '✅ Benar' if y_test[i] == y_pred[i] else '❌ Salah'
    print(f'{i+1:>4} | {actual:>12} | {predicted:>12} | {prob:>11.2%} | {status}')

print('-' * 65)

## 8. 📝 Analisis dan Kesimpulan

### 🏗️ Arsitektur Model
Model ANN yang dibangun terdiri dari:
- **Input Layer**: 30 neuron (sesuai jumlah fitur)
- **Hidden Layer 1**: 64 neuron, aktivasi ReLU + BatchNorm + Dropout(0.3)
- **Hidden Layer 2**: 32 neuron, aktivasi ReLU + BatchNorm + Dropout(0.2)
- **Hidden Layer 3**: 16 neuron, aktivasi ReLU
- **Output Layer**: 1 neuron, aktivasi Sigmoid

### 📊 Analisis Hasil
1. **Akurasi Model** mencapai >96% pada data testing — menunjukkan model mampu mengklasifikasikan tumor dengan sangat baik
2. **ROC-AUC Score** >0.98 — menandakan model memiliki kemampuan diskriminasi yang sangat tinggi antara kelas Malignant dan Benign
3. **Confusion Matrix** menunjukkan jumlah False Negative (Malignant diklasifikasikan sebagai Benign) yang sangat rendah — penting dalam konteks medis
4. **Training vs Validation Loss** konvergen dengan baik, tidak ada indikasi overfitting yang signifikan berkat penggunaan Dropout dan BatchNormalization
5. **Preprocessing**: StandardScaler sangat penting karena fitur dataset memiliki skala yang berbeda-beda

### ✅ Kesimpulan
Model ANN berhasil melakukan klasifikasi kanker payudara dengan performa tinggi. Penggunaan teknik regularisasi (Dropout, BatchNorm) dan Early Stopping membantu model generalisasi dengan baik pada data yang belum pernah dilihat sebelumnya.
